# 01 — LLM Evaluation: From Transformer Theory to Benchmarking

## Part 1: Theory

### 1.1 What Are Large Language Models?

LLMs are neural networks based on the **Transformer architecture** (Vaswani et al., 2017).

**Core Components:**
```
Input Text → Tokenizer → Token IDs → Embedding Layer → Transformer Blocks → Output Logits → Token Prediction
```

**Transformer Block internals:**
- **Multi-Head Self-Attention**: Computes relationships between ALL tokens simultaneously
  - Q (Query), K (Key), V (Value) matrices: `Attention(Q,K,V) = softmax(QK^T / √d_k) × V`
  - Multiple heads learn different relationship patterns (syntax, semantics, coreference)
- **Feed-Forward Network (FFN)**: 2-layer MLP that transforms each token independently
- **Layer Normalization + Residual Connections**: Stabilize training

**Tokenization** (how text becomes numbers):
| Method | Used By | Vocab Size | Example: "unhappiness" |
|--------|---------|-----------|------------------------|
| BPE | GPT, LLaMA | 32K-128K | ["un", "happiness"] |
| WordPiece | BERT | 30K | ["un", "##happi", "##ness"] |
| SentencePiece | T5, Gemma | 32K | ["▁un", "happi", "ness"] |

### 1.2 How LLMs Generate Text

**Autoregressive decoding**: Generate one token at a time, feeding output back as input.

```
P(token_t | token_1, token_2, ..., token_{t-1})
```

**Sampling parameters:**
- **Temperature (T)**: `softmax(logits / T)` — T=0 greedy, T=1 normal, T>1 creative
- **Top-p (nucleus)**: Sample from smallest set of tokens whose cumulative probability ≥ p
- **Top-k**: Only consider the k highest-probability tokens

### 1.3 Types of LLMs

```
Base Model (pre-trained on internet text)
    ↓ Supervised Fine-Tuning (SFT) on instruction-response pairs
Instruction-Tuned Model
    ↓ RLHF / DPO (human preference alignment)
Aligned Model (ChatGPT, Claude, etc.)
```

### 1.4 Evaluation Dimensions

| Dimension | What it measures | Why it matters |
|-----------|-----------------|----------------|
| **Accuracy** | Correctness of outputs | Core quality |
| **Latency** | Time to first token + generation time | User experience |
| **Throughput** | Tokens/second, requests/second | Scalability |
| **Cost** | $/million tokens | Business viability |
| **Safety** | Refusal of harmful content, bias | Compliance |

### 1.5 Evaluation Metrics Deep-Dive

| Metric | Formula/Method | Best For |
|--------|---------------|----------|
| **BLEU** | n-gram precision with brevity penalty | Translation |
| **ROUGE-L** | Longest common subsequence F1 | Summarization |
| **Perplexity** | exp(-1/N × Σ log P(token_i)) | Language modeling |
| **Semantic Similarity** | cosine(embed(output), embed(reference)) | Open-ended Q&A |
| **Human Eval** | Likert scale ratings by humans | Gold standard |

### 1.6 Major Benchmarks

- **MMLU** (57 subjects): General knowledge across domains
- **HumanEval** (164 problems): Code generation correctness
- **MT-Bench** (80 multi-turn): Conversational quality (GPT-4 as judge)
- **HELM** (42 scenarios): Holistic evaluation across many axes

---

### Architecture: LLM Inference Pipeline
```
┌──────────┐    ┌───────────┐    ┌─────────────────────────┐    ┌──────────┐    ┌────────────┐
│  Input   │───▶│ Tokenizer │───▶│  Transformer Layers ×N  │───▶│ Sampling │───▶│ Detokenize │
│  Text    │    │ (BPE)     │    │ (Attention + FFN + Norm) │    │ (T,p,k)  │    │  → Text    │
└──────────┘    └───────────┘    └─────────────────────────┘    └──────────┘    └────────────┘
```

### Architecture: Evaluation Framework
```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐     ┌────────────┐
│  Eval       │────▶│  Models Under    │────▶│  Metrics Engine │────▶│  Analysis  │
│  Dataset    │     │  Test (via API)  │     │  (similarity,   │     │  & Report  │
│  (Q&A pairs)│     │                  │     │   latency, cost)│     │  (charts)  │
└─────────────┘     └──────────────────┘     └─────────────────┘     └────────────┘
```

## Part 2: Implementation

### Setup

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy import stats

load_dotenv()

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")

# Free models to evaluate
models = {
    "LLaMA-3.1-8B": ChatOpenAI(
        model="meta-llama/llama-3.1-8b-instruct:free",
        openai_api_key=OPENROUTER_KEY,
        openai_api_base=OPENROUTER_BASE,
        temperature=0,
    ),
    "Gemma-2-9B": ChatOpenAI(
        model="google/gemma-2-9b-it:free",
        openai_api_key=OPENROUTER_KEY,
        openai_api_base=OPENROUTER_BASE,
        temperature=0,
    ),
}

# Free local embedding model for scoring
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Models: {list(models.keys())}\nEmbedding dim: {embedder.get_sentence_embedding_dimension()}")

### Evaluation Dataset

In [ ]:
eval_dataset = [
    {"question": "What is retrieval-augmented generation?",
     "ground_truth": "RAG combines information retrieval with text generation, allowing LLMs to access external knowledge bases to produce more accurate and grounded responses.",
     "category": "factual"},
    {"question": "Explain the difference between fine-tuning and prompt engineering.",
     "ground_truth": "Fine-tuning modifies model weights on domain-specific data, while prompt engineering crafts input instructions to guide model behavior without changing weights.",
     "category": "comparison"},
    {"question": "What are vector embeddings used for in AI systems?",
     "ground_truth": "Vector embeddings represent text as dense numerical vectors enabling semantic similarity search, clustering, and retrieval in AI applications.",
     "category": "factual"},
    {"question": "How does the transformer attention mechanism work?",
     "ground_truth": "Attention computes weighted relationships between all tokens using query, key, and value matrices, allowing the model to focus on relevant context regardless of position.",
     "category": "explanation"},
    {"question": "What is model drift and how do you detect it?",
     "ground_truth": "Model drift is performance degradation over time due to changing data distributions. Detected by monitoring prediction distributions, accuracy metrics, and statistical tests.",
     "category": "explanation"},
    {"question": "Compare serverless vs container deployment for AI.",
     "ground_truth": "Serverless (Lambda) offers auto-scaling and zero ops but has cold starts and size limits. Containers (ECS/K8s) offer more control, GPU support, but require infrastructure management.",
     "category": "comparison"},
    {"question": "What is RLHF?",
     "ground_truth": "Reinforcement Learning from Human Feedback trains a reward model on human preferences, then uses PPO to optimize the LLM policy to generate outputs humans prefer.",
     "category": "factual"},
    {"question": "Explain the CAP theorem in distributed systems.",
     "ground_truth": "CAP theorem states a distributed system can only guarantee two of three: Consistency, Availability, and Partition tolerance. Most systems choose AP or CP.",
     "category": "explanation"},
    {"question": "What is the difference between BLEU and ROUGE metrics?",
     "ground_truth": "BLEU measures precision of n-grams in generated text against references. ROUGE measures recall, checking how much of the reference appears in the generated text.",
     "category": "comparison"},
    {"question": "How does top-p sampling work?",
     "ground_truth": "Top-p (nucleus) sampling selects from the smallest set of tokens whose cumulative probability exceeds p, dynamically adjusting the candidate pool size per step.",
     "category": "explanation"},
]

print(f"Evaluation dataset: {len(eval_dataset)} questions")
print(f"Categories: {pd.Series([d['category'] for d in eval_dataset]).value_counts().to_dict()}")

### Run Evaluation

In [ ]:
def semantic_similarity(text1: str, text2: str) -> float:
    emb = embedder.encode([text1, text2])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

def evaluate_model(model_name, model, dataset):
    results = []
    for item in dataset:
        start = time.time()
        response = model.invoke(item["question"])
        latency = time.time() - start
        answer = response.content if hasattr(response, "content") else str(response)
        sim = semantic_similarity(answer, item["ground_truth"])
        results.append({
            "model": model_name,
            "question": item["question"][:50],
            "category": item["category"],
            "similarity": round(sim, 4),
            "latency_s": round(latency, 3),
            "response_length": len(answer),
            "tokens_approx": len(answer) // 4,
        })
    return results

all_results = []
for name, model in models.items():
    print(f"Evaluating {name}...")
    all_results.extend(evaluate_model(name, model, eval_dataset))

df = pd.DataFrame(all_results)
print(f"\nCompleted: {len(df)} evaluations")
df.head()

### Results Analysis & Visualization

In [ ]:
summary = df.groupby("model").agg(
    avg_similarity=("similarity", "mean"),
    avg_latency=("latency_s", "mean"),
    p95_latency=("latency_s", lambda x: np.percentile(x, 95)),
    avg_tokens=("tokens_approx", "mean"),
    throughput_qps=("latency_s", lambda x: 1 / x.mean()),
).round(4)
print("=== Model Comparison ===")
print(summary)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.boxplot(data=df, x="model", y="similarity", ax=axes[0,0])
axes[0,0].set_title("Semantic Similarity (Accuracy)")
axes[0,0].set_ylim(0, 1)

sns.boxplot(data=df, x="model", y="latency_s", ax=axes[0,1])
axes[0,1].set_title("Latency Distribution (seconds)")

sns.barplot(data=df, x="category", y="similarity", hue="model", ax=axes[1,0])
axes[1,0].set_title("Accuracy by Category")

axes[1,1].bar(summary.index, summary["throughput_qps"])
axes[1,1].set_title("Throughput (queries/sec)")
axes[1,1].set_ylabel("QPS")

plt.tight_layout()
plt.show()

### Statistical Significance Testing

In [ ]:
model_names = list(models.keys())
scores_a = df[df["model"] == model_names[0]]["similarity"].values
scores_b = df[df["model"] == model_names[1]]["similarity"].values

# Paired t-test
t_stat, p_value = stats.ttest_rel(scores_a, scores_b)
print(f"Paired t-test: t={t_stat:.4f}, p={p_value:.4f}")
print(f"  → {'Significant' if p_value < 0.05 else 'Not significant'} difference (α=0.05)")

# Wilcoxon signed-rank (non-parametric alternative)
w_stat, w_p = stats.wilcoxon(scores_a, scores_b)
print(f"\nWilcoxon: W={w_stat:.4f}, p={w_p:.4f}")

# Effect size (Cohen's d)
cohen_d = (scores_a.mean() - scores_b.mean()) / np.sqrt((scores_a.std()**2 + scores_b.std()**2) / 2)
print(f"\nCohen's d: {cohen_d:.4f} ({'small' if abs(cohen_d)<0.5 else 'medium' if abs(cohen_d)<0.8 else 'large'} effect)")

# 95% Confidence interval for difference
diff = scores_a - scores_b
ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
print(f"95% CI for difference: [{ci[0]:.4f}, {ci[1]:.4f}]")

## Part 3: Key Takeaways

1. **Semantic similarity** is the best automated metric for open-ended Q&A (correlates with human judgment)
2. **Statistical tests** prevent drawing conclusions from noise — always check significance
3. **Category-level analysis** reveals model strengths (some excel at factual, others at reasoning)
4. **Latency varies** significantly — P95 matters more than average for user experience
5. **Free models** (LLaMA, Gemma) are competitive for many tasks

### Next → 02_model_selection.ipynb